In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
!pip uninstall -y transformers accelerate peft datasets trl bitsandbytes
!pip install -q \
  "transformers==4.46.3" \
  "accelerate==1.0.1" \
  "peft==0.13.2" \
  "datasets==3.1.0" \
  "scikit-learn" \
  "sentencepiece"

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: accelerate 1.0.1
Uninstalling accelerate-1.0.1:
  Successfully uninstalled accelerate-1.0.1
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: datasets 3.1.0
Uninstalling datasets-3.1.0:
  Successfully uninstalled datasets-3.1.0


In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from peft import LoraConfig, get_peft_model, TaskType

from dataclasses import dataclass
from typing import Optional, Dict, Any, List

from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from sklearn.linear_model import LinearRegression

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_CSV = "/content/ielts_train_df.csv"
VAL_CSV   = "/content/ielts_val_df.csv"
TEST_CSV  = "/content/ielts_test_df.csv"

MAX_LENGTH = 768
SEED = 42

BATCH_SIZE = 1
GRAD_ACCUM = 16

LR = 3e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01
HEAD_DROPOUT = 0.1
GRA_LOSS_WEIGHT = 1.0
BIAS_LOSS_WEIGHT = 0.02
OUTPUT_DIR = "./qwen25_3b_ielts_multitask"

TARGET_COLS = ["TR", "CC", "LR", "GRA"]
REG_TARGET_COLS = ["TR", "CC", "LR"]
GRA_CLASSES = np.arange(4.0, 9.0 + 0.5, 0.5, dtype=np.float32)  # 4.0, 4.5, ..., 9.0

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [4]:
# --- CẬP NHẬT CELL 3: LÀM SẠCH DỮ LIỆU TRIỆT ĐỂ ---

# 1. Load dữ liệu gốc
train_df = pd.read_csv(TRAIN_CSV, engine="python", on_bad_lines="skip")
val_df = pd.read_csv(VAL_CSV, engine="python", on_bad_lines="skip") if os.path.exists(VAL_CSV) else None
test_df = pd.read_csv(TEST_CSV, engine="python", on_bad_lines="skip") if os.path.exists(TEST_CSV) else None

# 2. Tách tập validation nếu chưa có
if val_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

needed_cols = ["prompt", "essay"] + TARGET_COLS

def robust_clean_df(df):
    if df is None:
        return None

    df = df[needed_cols].copy()

    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["essay"] = df["essay"].astype(str).str.strip()

    df = df[(df["prompt"].str.len() > 0) & (df["essay"].str.len() > 20)].reset_index(drop=True)

    for col in TARGET_COLS:
        df[col] = df[col].clip(0.0, 9.0)

    return df

# Áp dụng hàm làm sạch cho các tập dữ liệu
train_df = robust_clean_df(train_df)
val_df = robust_clean_df(val_df)
test_df = robust_clean_df(test_df)

print(f"Train shape: {train_df.shape if train_df is not None else 0}")
print(f"Val shape: {val_df.shape if val_df is not None else 0}")
print(train_df[TARGET_COLS].head())

Train shape: (6618, 6)
Val shape: (827, 6)
    TR   CC   LR  GRA
0  6.0  6.0  6.0  6.0
1  6.5  6.5  6.5  6.5
2  7.0  7.0  7.0  7.0
3  5.0  5.0  5.0  5.0
4  4.5  4.5  4.5  4.5


In [5]:
def build_input_text(row):
    prompt = str(row["prompt"]).strip()
    essay = str(row["essay"]).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

train_df["text"] = train_df.apply(build_input_text, axis=1)
val_df["text"] = val_df.apply(build_input_text, axis=1)

if test_df is not None:
    test_df["text"] = test_df.apply(build_input_text, axis=1)

def make_labels(df):
    df = df.copy()
    df["labels"] = (df[TARGET_COLS] / 9.0).values.tolist()
    return df

train_df = make_labels(train_df)
val_df = make_labels(val_df)
if test_df is not None:
    test_df = make_labels(test_df)

In [6]:
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

if test_df is not None:
    test_ds = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
    dataset_dict["test"] = test_ds

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format(type="torch")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6618 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

In [8]:
from types import SimpleNamespace

class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        self.backbone = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        self.backbone.config.use_cache = False

        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        )
        self.backbone = get_peft_model(self.backbone, lora_config)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.head_tr = nn.Linear(hidden_size, 1)
        self.head_cc = nn.Linear(hidden_size, 1)
        self.head_lr = nn.Linear(hidden_size, 1)
        self.head_gra = nn.Linear(hidden_size, len(GRA_CLASSES))

        self.reg_activation = nn.Sigmoid()
        self.register_buffer("gra_class_values", torch.tensor(GRA_CLASSES, dtype=torch.float32))

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )
        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        tr = self.reg_activation(self.head_tr(pooled))
        cc = self.reg_activation(self.head_cc(pooled))
        lr = self.reg_activation(self.head_lr(pooled))
        reg_logits = torch.cat([tr, cc, lr], dim=1)

        gra_logits = self.head_gra(pooled)
        gra_probs = torch.softmax(gra_logits, dim=-1)
        gra_values = self.gra_class_values.to(dtype=gra_logits.dtype)
        gra_score = (gra_probs * gra_values.unsqueeze(0)).sum(dim=-1, keepdim=True) / 9.0

        combined_logits = torch.cat([reg_logits, gra_score], dim=1)

        return {
            "logits": combined_logits,
            "reg_logits": reg_logits,
            "gra_logits": gra_logits,
        }

model = QwenForIELTSMultiTask(MODEL_NAME, tokenizer, head_dropout=HEAD_DROPOUT)
model.backbone.print_trainable_parameters()
print("Heads are trainable by default.")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
Heads are trainable by default.


In [9]:
class IELTSMultiTaskTrainer(Trainer):
    def __init__(self, *args, gra_loss_weight=1.0, bias_loss_weight=0.02, **kwargs):
        super().__init__(*args, **kwargs)
        self.gra_loss_weight = gra_loss_weight
        self.bias_loss_weight = bias_loss_weight
        self.reg_loss_fn = nn.HuberLoss(delta=0.08, reduction="none")
        self.gra_loss_fn = nn.CrossEntropyLoss()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")   # normalized labels: [TR, CC, LR, GRA] in [0, 1]
        outputs = model(**inputs)

        reg_logits = outputs["reg_logits"]
        gra_logits = outputs["gra_logits"]

        labels = labels.to(reg_logits.dtype)
        reg_labels = labels[:, :3]

        gra_scores = (labels[:, 3] * 9.0).float()
        gra_class = torch.round((gra_scores - 4.0) * 2).long().clamp(
            min=0, max=len(GRA_CLASSES) - 1
        )

        reg_loss_per_dim = self.reg_loss_fn(reg_logits, reg_labels)   # (B, 3)
        reg_loss = reg_loss_per_dim.mean()   # thay vì sum(dim=1).mean()

        gra_loss = self.gra_loss_fn(gra_logits, gra_class)

        pred_mean = outputs["logits"].mean(dim=0)
        label_mean = labels.mean(dim=0)
        bias_loss = torch.abs(pred_mean - label_mean).mean()

        loss = reg_loss + self.gra_loss_weight * gra_loss + self.bias_loss_weight * bias_loss
        return (loss, outputs) if return_outputs else loss


In [10]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class RegressionCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.stack([
            f["labels"] if isinstance(f["labels"], torch.Tensor)
            else torch.tensor(f["labels"], dtype=torch.float)
            for f in features
        ]).float()

        batch = self.tokenizer.pad(
            [{k: v for k, v in f.items() if k != "labels"} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        return batch

data_collator = RegressionCollator(tokenizer)

In [11]:
def round_to_half(x):
    return np.round(x * 2) / 2

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.float32)

    preds = preds * 9.0
    labels = labels * 9.0

    preds = np.clip(preds, 0.0, 9.0)
    labels = np.clip(labels, 0.0, 9.0)

    preds_half = round_to_half(preds)

    maes = []
    qwks = []
    biases = []

    for i in range(labels.shape[1]):
        maes.append(mean_absolute_error(labels[:, i], preds_half[:, i]))
        biases.append(float(np.mean(preds_half[:, i] - labels[:, i])))

        y_true = np.round(labels[:, i] * 2).astype(int)
        y_pred = np.round(preds_half[:, i] * 2).astype(int)
        qwks.append(cohen_kappa_score(y_true, y_pred, weights="quadratic"))

    within_05 = np.mean(np.abs(preds_half - labels) <= 0.5)

    return {
        "mean_mae": float(np.mean(maes)),
        "mean_qwk": float(np.mean(qwks)),
        "tr_qwk": float(qwks[0]),
        "cc_qwk": float(qwks[1]),
        "lr_qwk": float(qwks[2]),
        "gra_qwk": float(qwks[3]),
        "tr_bias": float(biases[0]),
        "cc_bias": float(biases[1]),
        "lr_bias": float(biases[2]),
        "gra_bias": float(biases[3]),
        "within_0.5_acc": float(within_05),
    }


In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="gra_qwk",
    greater_is_better=True,

    bf16=torch.cuda.is_available(),
    fp16=False,

    gradient_checkpointing=False,
    remove_unused_columns=False,
    report_to="none",
    save_total_limit=2,
    max_grad_norm=1.0,
    label_names=["labels"],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = max(
    1, math.ceil(len(tokenized_ds["train"]) / (BATCH_SIZE * GRAD_ACCUM))
)

num_training_steps = num_update_steps_per_epoch * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps,
)

print("num_update_steps_per_epoch =", num_update_steps_per_epoch)
print("num_training_steps =", num_training_steps)


num_update_steps_per_epoch = 414
num_training_steps = 1242


In [14]:
trainer = IELTSMultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    optimizers=(optimizer, scheduler),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    gra_loss_weight=GRA_LOSS_WEIGHT,
    bias_loss_weight=BIAS_LOSS_WEIGHT,
)

In [15]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(debug_loader))
print(batch.keys())
print(batch["labels"])
print(batch["labels"].shape)
print(batch["labels"].dtype)

KeysView({'input_ids': tensor([[ 42347,   3361,   2828,    921,   8441,   5837,    525,  10164,    264,
           6765,   3311,    315,   3220,    389,  12613,    862,  27550,    311,
           1896,    949,    304,   1045,  15245,   9833,  42582,     13,  25028,
          17585,    429,    432,   1035,    387,   2664,    421,   1493,   5837,
            646,   8329,    279,   3220,    389,   2841,    311,   1896,    949,
            304,   9833,     13,   2014,   1128,  12818,    653,    498,   7503,
            476,  28295,   1939,     58,   9996,   3022,    921,   3862,    525,
           2696,    315,   5837,    304,    279,   1879,    879,    525,   1667,
           2244,   3311,    315,   3220,    389,  42649,    315,    862,   7263,
             13,    358,   7503,    429,    807,   1265,   1191,  10164,   3220,
            389,   2841,  10552,   7488,    304,    429,   1616,    807,    686,
           1191,    279,  50375,    315,   4124,  17628,    304,    862,   2272,
     

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:", val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
    print("Test metrics:", test_metrics)

# ===== Optional calibration on validation set =====
val_pred_output = trainer.predict(tokenized_ds["validation"])
val_preds = np.asarray(val_pred_output.predictions, dtype=np.float32) * 9.0
val_labels = np.asarray(val_pred_output.label_ids, dtype=np.float32) * 9.0

val_preds = np.clip(val_preds, 0.0, 9.0)
val_labels = np.clip(val_labels, 0.0, 9.0)

calibration_models = {}
for i, col in enumerate(TARGET_COLS):
    lr_model = LinearRegression()
    lr_model.fit(val_preds[:, i].reshape(-1, 1), val_labels[:, i])
    calibration_models[col] = {
        "a": float(lr_model.coef_[0]),
        "b": float(lr_model.intercept_),
    }

print("Calibration params:")
print(calibration_models)


In [ ]:
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

OUTPUT_DIR = "./qwen25_3b_ielts_multitask"

shutil.make_archive("/content/model_output", 'zip', OUTPUT_DIR)

shutil.copy("/content/model_output.zip",
            "/content/drive/MyDrive/model_output.zip")

print("Model zipped and saved to Google Drive!")

In [ ]:
import os
import gc
import torch

# Nếu muốn train tiếp thêm, tăng tổng số epoch lên
# Ví dụ trước đó đã train 3 epoch, giờ muốn train tới epoch 6:
trainer.args.num_train_epochs = 8

# Tự tìm checkpoint mới nhất trong OUTPUT_DIR
checkpoint_dirs = [
    os.path.join(OUTPUT_DIR, d)
    for d in os.listdir(OUTPUT_DIR)
    if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
]

if not checkpoint_dirs:
    raise ValueError(f"Không tìm thấy checkpoint nào trong {OUTPUT_DIR}")

latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
print("Resuming from:", latest_checkpoint)

gc.collect()
torch.cuda.empty_cache()

train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

print("Resume train xong.")
print(train_result)